# CIÊNCIA DE DADOS - DCA3501

UNIVERSIDADE FEDERAL DO RIO GRANDE DO NORTE, NATAL/RN

DEPARTAMENTO DE ENGENHARIA DE COMPUTAÇÃO E AUTOMAÇÃO

### Projeto - Mini-aula em Jupyter Notebook sobre Técnicas de Aprendizado de Máquina 

**Preencha abaixo com as informações da sua mini-aula.**

## A. Identificação

- Título da técnica e da mini-aula: 
- Nome(s) do(s) aluno(s): 
- Matrícula(s): 


## B. Introdução e motivação

Explique, em linguagem simples:

- Que tipo de problema a técnica resolve (classificação, regressão, agrupamento, redução de dimensionalidade etc.);
- Em que contextos ela é usada (exemplos de aplicações reais).


## C. Fundamentos teóricos básicos

Apresente a intuição da técnica e, quando pertinente:

- Explique a ideia geral (como o modelo "pensa");
- Inclua expressões matemáticas (é possível usar LaTeX para exibir equações);
- Relacione com conceitos já vistos na disciplina (distância, regressão, hiperplano, variância, etc.).

Use este espaço para escrever a teoria em Markdown. Você pode usar fórmulas como, por exemplo:  
`$y = w^T x + b$`.


## D. Descrição do dataset

Descreva o conjunto de dados que será utilizado:

- Fonte do dataset (sklearn, Kaggle, UCI, arquivo próprio, etc.);
- Número de instâncias (linhas) e variáveis (colunas);
- Descrição das principais features e do alvo (se houver);
- Possíveis desafios dos dados (desbalanceamento, ruídos, outliers, etc.).


Utilizamos o **Wine dataset**, obtido do **Kaggle** (arquivo `Wine_dataset.csv`). É uma cópia do clássico *Wine recognition dataset* do **UCI Machine Learning Repository**, construído a partir da análise química de vinhos produzidos numa mesma região da Itália, porém derivados de **três cultivares** (variedades de uva) distintos. Por ter um alvo categórico com mais de duas classes, é um problema de **classificação multiclasse**, amplamente usado como *benchmark*.

### Dimensões
- **178 instâncias** (linhas) — cada uma é uma amostra de vinho analisada em laboratório;
- **14 colunas** — **13 features** numéricas (medidas físico-químicas) e **1 variável-alvo** (`class`);
- **Tipos:** 11 colunas em ponto flutuante e 3 inteiras (`class`, `Magnesium` e `Proline`); todas numéricas, sem variáveis de texto a codificar;
- **Valores ausentes:** nenhum — as 178 amostras estão completas em todas as colunas.

A célula abaixo carrega o arquivo e exibe uma amostra das primeiras linhas. O cabeçalho do CSV traz a última coluna como `"Proline "` (com um espaço final), então limpamos os nomes das colunas com `str.strip()` para evitar `KeyError` ao referenciá-las depois.

In [1]:
import pandas as pd

df = pd.read_csv("Wine_dataset.csv")
df.columns = df.columns.str.strip()   # remove o espaço final de "Proline "
df.head()

,class,Alcohol,Malic acid,Ash,Alcalinity of ash,Magnesium,Total phenols,Flavanoids,Nonflavanoid phenols,Proanthocyanins,Color intensity,Hue,OD280/OD315 of diluted wines,Proline
0,1,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,1,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,1,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,1,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735


### As features

As 13 features são medidas físico-químicas obtidas em laboratório a partir de cada amostra:

| Feature | O que representa |
|---|---|
| `Alcohol` | Teor alcoólico do vinho |
| `Malic acid` | Ácido málico — ácido orgânico que influencia a acidez e o sabor |
| `Ash` | Cinzas — resíduo mineral inorgânico da amostra |
| `Alcalinity of ash` | Alcalinidade das cinzas (basicidade dos minerais) |
| `Magnesium` | Teor de magnésio |
| `Total phenols` | Fenóis totais — compostos ligados a cor, sabor e adstringência |
| `Flavanoids` | Flavonoides — subclasse de fenóis com ação antioxidante |
| `Nonflavanoid phenols` | Fenóis que não são flavonoides |
| `Proanthocyanins` | Proantocianinas — taninos condensados |
| `Color intensity` | Intensidade da cor do vinho |
| `Hue` | Matiz/tonalidade da cor |
| `OD280/OD315 of diluted wines` | Razão de absorbância óptica (280/315 nm) de vinhos diluídos |
| `Proline` | Prolina — aminoácido mais abundante no vinho |

### A variável-alvo (`class`)

O alvo identifica o **cultivar de origem** de cada vinho e assume três valores — **1, 2 e 3**. A distribuição é:

| Classe (cultivar) | Amostras | Proporção |
|---|---:|---:|
| 1 | 59 | 33,1% |
| 2 | 71 | 39,9% |
| 3 | 48 | 27,0% |

Como há três categorias, trata-se de um problema de **classificação multiclasse**.

### Possíveis desafios dos dados

**1. Desbalanceamento leve.** As classes não têm o mesmo tamanho: a classe 2 é a maior (71 amostras, ~40%) e a classe 3 a menor (48 amostras, 27%). Não é um desbalanceamento severo, mas é suficiente para justificar duas decisões: separar treino/teste de forma **estratificada** (Seção E) e, na avaliação, observar métricas por classe (precisão, recall e F1 com média *macro*), e não apenas a acurácia global — que pode mascarar um desempenho ruim na classe minoritária.

**2. Escalas muito diferentes.** Como mostrado na tabela de estatísticas, `Proline` opera na casa das centenas/milhares enquanto `Hue` e `Nonflavanoid phenols` ficam abaixo de 2. Para o SVM isso é crítico, pois o algoritmo se baseia em **distâncias entre pontos**: sem tratamento, as features de escala grande dominariam o cálculo e as de escala pequena seriam praticamente ignoradas. Esse desafio é resolvido com **padronização** na Seção E.

**3. Outliers pontuais.** Aplicando a regra do IQR (valores além de 1,5·IQR dos quartis), encontramos poucos valores extremos, concentrados em algumas features: `Alcalinity of ash` (4), `Magnesium` (4), `Color intensity` (4), `Malic acid` (3), `Ash` (3), `Proanthocyanins` (2) e `Hue` (1) — cerca de 21 ocorrências no total, num conjunto de 178x13 valores. Não chegam a comprometer o conjunto, mas reforçam a importância da padronização; caso fossem mais agressivos, poderíamos optar por um `RobustScaler`.

**4. Multicolinearidade entre features.** Várias medidas estão fortemente correlacionadas, sobretudo as ligadas a compostos fenólicos. Os pares mais correlacionados são `Flavanoids` <-> `Total phenols` (0,86), `Flavanoids` <-> `OD280/OD315` (0,79) e `OD280/OD315` <-> `Total phenols` (0,70). Isso é esperado quimicamente, pois medem aspectos relacionados dos mesmos compostos. A redundância afeta principalmente a **interpretabilidade** e modelos lineares; para o SVM com kernel RBF o impacto é menor, mas é uma característica importante de se conhecer.

**5. Conjunto pequeno e de baixo ruído.** São apenas 178 amostras para 13 dimensões. Isso traz dois efeitos: as métricas de teste podem **variar bastante** conforme o *split* — o que reforça o uso de **validação cruzada** nos experimentos (Tarefa 4) — e, por outro lado, o dataset é conhecido por ser "bem comportado", com classes bem separáveis e acurácias tipicamente altas. Ou seja, o desafio aqui não está tanto na dificuldade do problema, mas em **demonstrar bem a técnica** e analisar criticamente seu comportamento.

## E. Preparação dos dados

Descreva e implemente as etapas de pré-processamento:

- Divisão em treino e teste (ou validação cruzada);
- Normalização/padronização, se necessária;
- Tratamento de valores ausentes;
- Codificação de variáveis categóricas, se houver.


In [ ]:
# Exemplo de pré-processamento (ajuste ao seu caso)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Exemplo genérico: separe X e y
# X = df.drop(columns=["alvo"])
# y = df["alvo"]

                
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Exemplo de normalização (se necessário)
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

print("Implemente aqui a divisão treino/teste e o pré-processamento necessário.")

## F. Implementação do modelo

Nesta seção, você deve:

- Criar o modelo correspondente à técnica escolhida (por exemplo, `KNeighborsClassifier`, `SVC`, `MLPClassifier`, `DBSCAN`, etc.);
- Explicar, em texto, os principais hiperparâmetros utilizados;
- Treinar o modelo com os dados de treino.


In [ ]:
# Exemplo genérico de criação e treino de modelo (substitua pelo seu)
# from sklearn.neighbors import KNeighborsClassifier

# modelo = KNeighborsClassifier(n_neighbors=5)
# modelo.fit(X_train_scaled, y_train)

print("Crie e treine aqui o modelo da técnica escolhida.")

## G. Avaliação do modelo

Aqui você deve:

- Calcular métricas adequadas ao tipo de problema (acurácia, precisão, recall, F1, MAE, MSE, R², silhouette score, etc.);
- Produzir gráficos de avaliação, como matriz de confusão, curvas, visualização de clusters ou projeções 2D/3D.


In [ ]:
# Exemplo genérico de avaliação (ajuste ao seu problema)
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Exemplo para classificação (substitua pelas suas variáveis)
# y_pred = modelo.predict(X_test_scaled)
# print("Acurácia:", accuracy_score(y_test, y_pred))
# print(classification_report(y_test, y_pred))

# Exemplo de matriz de confusão
# cm = confusion_matrix(y_test, y_pred)
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
# plt.xlabel("Predito")
# plt.ylabel("Real")
# plt.show()

print("Implemente aqui as métricas e gráficos de avaliação apropriados.")

## H. Discussão e limitações

Discuta, em texto:

- Em que situações a técnica tende a funcionar bem;
- Quando ela pode apresentar dificuldades (ex: muitos atributos, poucos dados, ruído, etc.);
- Sensibilidade a hiperparâmetros e ao pré-processamento;
- Possíveis melhorias e trabalhos futuros (ex: testar outros modelos, ajustar hiperparâmetros, usar outro dataset).


## I. Conclusões

Resuma os principais pontos da mini-aula:

- O que vocês aprenderam sobre a técnica;
- Principais resultados obtidos; 
- Quando e por que recomendariam o uso desse método em um projeto real.


## J. Referências

Liste aqui as principais fontes usadas no trabalho (em formato livre, mas organizado):

- Documentações oficiais (por exemplo, scikit-learn, bibliotecas adicionais);
- Livros e artigos;
- Tutoriais e materiais complementares confiáveis.
